In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import DoubleType, IntegerType, LongType, StructField, StructType

MINIO_ROOT_USER = os.environ.get("MINIO_ROOT_USER", "minio")
MINIO_ROOT_PASSWORD = os.environ.get("MINIO_ROOT_PASSWORD", "miniominio")
SPARK_MINIO_ENDPOINT = os.environ.get(
    "SPARK_MINIO_ENDPOINT", "http://urbangreen-minio:9000"
)
KAFKA_TOPIC_SENSOR_READINGS = os.environ.get(
    "KAFKA_TOPIC_SENSOR_READINGS", "sensor_readings"
)
SIMULATOR_KAFKA_BOOTSTRAP = os.environ.get(
    "SIMULATOR_KAFKA_BOOTSTRAP", "urbangreen-kafka:9092"
)
SPARK_KAFKA_OFFSET = os.environ.get("SPARK_KAFKA_OFFSET", "earliest")

MINIO_STAGING_BUCKET = os.environ.get(
    "MINIO_STAGING_BUCKET", "staging"
)

STREAM_TRIGGER_INTERVAL = os.environ.get(
    "STREAM_TRIGGER_INTERVAL", "60 seconds"
)

In [2]:
# Start Spark Session
spark = (
    SparkSession.builder.appName("Sensor Readings Stream")
    .config("spark.hadoop.fs.s3a.endpoint", SPARK_MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ROOT_USER)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_ROOT_PASSWORD)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.sql.streaming.schemaInference", "false")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2",
            "org.apache.hadoop:hadoop-aws:3.4.1",
            "com.amazonaws:aws-java-sdk-bundle:1.12.782",
        ])
    )
    .getOrCreate()
)

print(spark.sparkContext._jsc.sc().listJars().toString())

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-486562e2-bf8c-4574-9b79-55ea549d3daf;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala

List(spark://cdb8b8198101:39467/jars/org.apache.kafka_kafka-clients-3.9.1.jar, spark://cdb8b8198101:39467/jars/org.apache.spark_spark-sql-kafka-0-10_2.13-4.0.2.jar, spark://cdb8b8198101:39467/jars/org.apache.hadoop_hadoop-client-runtime-3.4.1.jar, spark://cdb8b8198101:39467/jars/org.lz4_lz4-java-1.8.0.jar, spark://cdb8b8198101:39467/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.13-4.0.2.jar, spark://cdb8b8198101:39467/jars/com.google.code.findbugs_jsr305-3.0.0.jar, spark://cdb8b8198101:39467/jars/org.xerial.snappy_snappy-java-1.1.10.7.jar, spark://cdb8b8198101:39467/jars/org.apache.commons_commons-pool2-2.12.0.jar, spark://cdb8b8198101:39467/jars/org.apache.hadoop_hadoop-client-api-3.4.1.jar, spark://cdb8b8198101:39467/jars/org.slf4j_slf4j-api-2.0.16.jar, spark://cdb8b8198101:39467/jars/org.scala-lang.modules_scala-parallel-collections_2.13-1.2.0.jar)


In [3]:
SENSOR_SCHEMA = StructType(
    [
        StructField("farm_sensor_id", IntegerType(), True),
        StructField("farm_id", IntegerType(), True),
        StructField("sensor_type_id", IntegerType(), True),
        StructField("value", DoubleType(), True),
        StructField("timestamp", LongType(), True),
    ],
)

In [4]:
kafka_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", SIMULATOR_KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC_SENSOR_READINGS)
    .option("startingOffsets", SPARK_KAFKA_OFFSET)
    .option("failOnDataLoss", "false")
    .load()
)


In [5]:
raw_json_df = kafka_df.select(
    col("value").cast("string").alias("value")
)

In [6]:
parsed_df = raw_json_df.withColumn(
    "data",
    from_json(col("value"), SENSOR_SCHEMA)
)

In [7]:
parsed_df = parsed_df.select("data.*")

parsed_df_with_event_data = parsed_df.withColumn("event_date", to_date(from_unixtime(col("timestamp"))))

In [8]:
query = (
    parsed_df_with_event_data.writeStream
    .format("parquet")
    .outputMode("append")
    .partitionBy("event_date")
    .option(
        "path",
        f"s3a://{MINIO_STAGING_BUCKET}/raw/kafka/{KAFKA_TOPIC_SENSOR_READINGS}/",
    )
    .option(
        "checkpointLocation",
        f"s3a://{MINIO_STAGING_BUCKET}/_checkpoints/kafka/{KAFKA_TOPIC_SENSOR_READINGS}/",
    )
    .trigger(processingTime=STREAM_TRIGGER_INTERVAL)
    .start()
)

query.awaitTermination()

Py4JJavaError: An error occurred while calling o90.start.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2737)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3569)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource.makeQualified(DataSource.scala:125)
	at org.apache.spark.sql.execution.datasources.DataSource.createSink(DataSource.scala:332)
	at org.apache.spark.sql.classic.DataStreamWriter.createV1Sink(DataStreamWriter